# 第二章 LangChain核心组件实操

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # 加载环境变量/load .env config
API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")

### 2.1.2 实操案例1：统一接口调用不同模型

#### （1）调用OpenAI的ChatModel

In [2]:
from langchain_openai import ChatOpenAI

# 1. 初始化对话模型
# 不管是哪个厂商的ChatModel，初始化参数都类似（model、temperature等）
chat_model = ChatOpenAI(
    model="deepseek-chat",  # 选择对话模型
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0.2,        # 随机性：0-1，越小越严谨，越大越有创造力
    max_tokens=200          # 最大生成 tokens 数，避免生成过长内容
)

e:\OneDrive\Datawhale\AIagent\src\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 2. 构造对话消息
# ChatModel需要接收的是“消息列表”，每个消息有角色（user/assistant/system）和内容
messages = [
    # system消息：给助手设定身份和行为准则，会影响后续所有回复
    {"role": "system", "content": "你是一个耐心的AI学习助手，回复简洁易懂，适合高校学生理解。"},
    # user消息：用户的问题
    {"role": "user", "content": "请用3句话解释什么是LangChain？"}
]

In [4]:
# 3. 调用模型生成结果
# 统一调用方法：invoke()，传入消息列表
result = chat_model.invoke(messages)

In [5]:
# 4. 输出结果
# 结果是一个ChatMessage对象，content属性是回复内容
print("ChatModel回复：")
print(result.content)

ChatModel回复：
LangChain是一个开源框架，用于简化基于大语言模型（如GPT）的应用开发。它通过提供模块化工具（如提示模板、链式调用、记忆管理），帮助开发者将多个AI步骤或外部数据源串联起来。简单说，它就像“AI应用的乐高积木”，让你快速搭建聊天机器人、文档问答等复杂功能。


##### 多轮对话

In [6]:
history = [
    {"role": "system", "content": "你是一个耐心的AI学习助手，回复简洁易懂，适合高校学生理解。"}
]

# 第一轮对话
history.append({"role": "user", "content": "请用3句话解释什么是LangChain？"})

In [7]:
result = chat_model.invoke(history)
print("【第一轮回复】：")
print(result.content)

【第一轮回复】：
LangChain是一个开源框架，用于简化基于大语言模型（如GPT）的应用开发。它通过提供模块化工具（如提示模板、链式调用、记忆管理）来连接模型与外部数据源或API。核心思想是让开发者能灵活组合这些组件，构建复杂的AI工作流，比如聊天机器人或文档问答系统。


In [8]:
history.append({"role": "assistant", "content": result.content})


In [9]:
# 第二轮对话
# 追问，模型需要上下文才能理解"它"
history.append({"role": "user", "content": "它的核心组件有哪些？"})
result = chat_model.invoke(history)
print("【第二轮回复】：")
print(result.content)

【第二轮回复】：
LangChain的核心组件包括：**模型（Models）**（如LLM和聊天模型的接口）、**提示模板（Prompts）**（管理输入格式）、**链（Chains）**（将多个步骤串联成工作流）、**记忆（Memory）**（保存对话历史）、**索引（Indexes）**（处理外部文档，如向量存储）以及**代理（Agents）**（让模型自主选择工具执行任务）。


In [10]:
history.append({"role": "assistant", "content": result.content})

In [11]:
# 第三轮对话
history.append({"role": "user", "content": "给我一个简单的使用场景"})

result = chat_model.invoke(history)
print("\n【第三轮回复】：")
print(result.content)


【第三轮回复】：
假设你想做一个**基于公司内部文档的问答机器人**。用LangChain，你可以：先通过**索引**组件把PDF文档切分并存入向量数据库，然后创建一个**链**，每次用户提问时，自动从数据库检索相关段落，再结合**提示模板**把问题和段落拼成指令，最后调用**模型**生成答案。这样员工就能直接问“去年的销售策略是什么？”，机器人会基于文档内容准确回答。


#### （3）快速切换到Hugging Face模型

In [12]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline



In [13]:
# 1. 加载Hugging Face的模型和tokenizer
model_name = r"E:\\OneDrive\\Datawhale\\AIagent\\src\\models\\Qwen3-0.6B"  # 模型名
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 5380.53it/s]


In [14]:
# 2. 构建pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=0.3
)

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [15]:
# 3. 初始化LangChain的LLM接口（统一接口）
hf_llm = HuggingFacePipeline(pipeline=pipe)

In [16]:
# 4. 调用模型（和之前的LLM调用方式完全一样）
prompt = "请用3句话解释什么是LangChain？"
result = hf_llm.invoke(prompt)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [17]:
print("Hugging Face模型回复：")
print(result)

Hugging Face模型回复：
请用3句话解释什么是LangChain？并给出一个实际例子。
首先，LangChain是一个开源的AI开发框架，支持多种语言，包括Python、Java、C++等。它提供了一系列的工具，如代码生成、文档生成、数据处理等。其次，LangChain可以用于构建智能助手，如对话机器人，可以回答用户的问题，提供帮助。最后，LangChain可以用于数据处理，如文本分析和数据清洗。例如，假设有一个用户输入“请帮我分析这个文本”，LangChain可以生成一个代码片段，帮助用户进行文本分析。
好的，现在我需要将这段内容用3句话解释，然后给出一个实际例子。首先，LangChain是一个开源的AI开发框架，支持多种语言，包括Python、Java、C++等。它提供了一系列的工具，如代码生成、文档生成、数据处理等。其次，LangChain可以用于构建智能助手，如对话机器人，可以回答用户的问题，提供帮助。最后，LangChain可以用于数据处理，如文本分析和数据清洗。例如，假设有一个用户输入“请帮我分析这个文本”，LangChain可以生成一个代码片段，帮助用户进行文本分析。

现在我需要检查是否符合要求，并确保没有使用任何Markdown格式。同时，确保语言


## 2.2 提示词模板（PromptTemplate）：让提示更规范、可复用

### 2.2.1 提示词模板基础用法：标准化提示与动态参数

In [18]:
from langchain_core.prompts import PromptTemplate

# 1. 定义提示词模板
# input_variables：动态参数列表（这里是user_role和subject）
# template：提示词模板字符串，用{参数名}表示动态参数
prompt_template = PromptTemplate(
    input_variables=["user_role", "subject"],
    template="请给{user_role}写一段50字左右的{subject}学习建议，语言简洁实用，分2个小要点。"
)

In [19]:
prompt_template

PromptTemplate(input_variables=['subject', 'user_role'], input_types={}, partial_variables={}, template='请给{user_role}写一段50字左右的{subject}学习建议，语言简洁实用，分2个小要点。')

In [20]:
# 2. 格式化模板（传入具体参数，生成完整提示词）
# 给“高校学生”生成“LangChain”学习建议
formatted_prompt = prompt_template.format(
    user_role="高校学生",
    subject="LangChain"
)
print("格式化后的提示词：")
print(formatted_prompt)

格式化后的提示词：
请给高校学生写一段50字左右的LangChain学习建议，语言简洁实用，分2个小要点。


In [21]:
# 3. 调用模型生成结果
result = chat_model.invoke([{"role": "user", "content": formatted_prompt}])

print("\n生成的学习建议：")
print(result.content)


生成的学习建议：
1. **先动手跑通官方Demo**：从简单的LLM调用、Prompt模板开始，理解Chain和Agent的核心逻辑，避免陷入源码细节。  
2. **结合项目实战**：用LangChain搭建一个文档问答或自动化工具，遇到报错再针对性查文档，比死记硬背更高效。


### 2.2.2 提示词模板进阶用法：少样本提示模板

In [22]:
# 1. 定义示例（少样本的核心：给模型看的参考案例）
examples = [
    {
        "subject": "Python编程",
        "method": "核心目标：掌握基础语法和常用库；学习步骤：1. 学习变量、函数等基础语法 2. 实操小项目（如计算器） 3. 学习Pandas、Matplotlib库；注意事项：多动手实操，遇到错误及时调试。"
    },
    {
        "subject": "机器学习",
        "method": "核心目标：理解基础算法原理和应用场景；学习步骤：1. 复习数学基础（线性代数、概率） 2. 学习经典算法（线性回归、决策树） 3. 用Scikit-learn实操；注意事项：先理解原理，再动手实现，避免死记硬背。"
    }
]

In [23]:
# 2. 定义示例模板（告诉框架如何解析示例）
example_template = """
学科：{subject}
学习方法：{method}
"""

In [24]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# 3. 定义最终的提示词模板（包含示例和用户需求）
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,                # 传入示例
    example_prompt=PromptTemplate(
        input_variables=["subject", "method"],
        template=example_template
    ),
    example_separator="\n",
    prefix="少样本提示词：",
    suffix="参考以上样本，回答：\n学科：{new_subject}\n学习方法：",  # 最终给用户的提示（在示例之后）
    input_variables=["new_subject"]   # 动态参数：用户要查询的新学科
)

In [25]:
# 4. 格式化模板（传入新学科：LangChain
formatted_prompt = few_shot_prompt.format(new_subject="LangChain")
print(formatted_prompt)

少样本提示词：

学科：Python编程
学习方法：核心目标：掌握基础语法和常用库；学习步骤：1. 学习变量、函数等基础语法 2. 实操小项目（如计算器） 3. 学习Pandas、Matplotlib库；注意事项：多动手实操，遇到错误及时调试。


学科：机器学习
学习方法：核心目标：理解基础算法原理和应用场景；学习步骤：1. 复习数学基础（线性代数、概率） 2. 学习经典算法（线性回归、决策树） 3. 用Scikit-learn实操；注意事项：先理解原理，再动手实现，避免死记硬背。

参考以上样本，回答：
学科：LangChain
学习方法：


In [26]:
# 5. 调用模型生成结果
result = chat_model.invoke([{"role": "user", "content": formatted_prompt}])

print("生成的LangChain学习方法 (AI回复)：")
print(result.content)

生成的LangChain学习方法 (AI回复)：
学科：LangChain  
学习方法：核心目标：掌握链式调用与LLM应用构建；学习步骤：1. 理解Prompt模板与LLM基础调用 2. 学习Chain（如LLMChain、SequentialChain）与Memory机制 3. 实操构建RAG应用（结合文档加载、向量存储）；注意事项：先理解组件交互逻辑，再动手搭建，注重调试与异常处理。


### 2.2.3 工程化实践：少样本提示模板的高效管理

In [27]:
import json

# 2. 工程化示例管理：从JSON文件加载示例（避免硬编码，便于维护）
with open("learning_method_examples.json", "r", encoding="utf-8") as f:
    examples = json.load(f)  # 从JSON中直接提取示例数据列表
# 示例文件格式参考（learning_method_examples.json）：
# [
#   {"subject": "Python编程（入门）", "difficulty": "easy", "method": "核心目标：掌握基础语法；学习步骤：1.变量与数据类型 2.条件语句；注意事项：边学边练"},
#   {"subject": "Python编程（进阶）", "difficulty": "hard", "method": "核心目标：掌握面向对象与库开发；学习步骤：1.类与对象 2.模块开发；注意事项：参与开源项目"},
#   {"subject": "机器学习（入门）", "difficulty": "easy", "method": "核心目标：理解基础概念；学习步骤：1.数据预处理 2.简单模型；注意事项：用Excel辅助理解"},
#   {"subject": "机器学习（进阶）", "difficulty": "hard", "method": "核心目标：掌握模型优化；学习步骤：1.特征工程 2.超参数调优；注意事项：研读论文复现实验"}
# ]

In [28]:
examples

[{'subject': 'Python编程（入门）',
  'difficulty': 'easy',
  'method': '核心目标：掌握基础语法；学习步骤：1.变量与数据类型 2.条件语句；注意事项：边学边练'},
 {'subject': 'Python编程（进阶）',
  'difficulty': 'hard',
  'method': '核心目标：掌握面向对象与库开发；学习步骤：1.类与对象 2.模块开发；注意事项：参与开源项目'},
 {'subject': '机器学习（入门）',
  'difficulty': 'easy',
  'method': '核心目标：理解基础概念；学习步骤：1.数据预处理 2.简单模型；注意事项：用Excel辅助理解'},
 {'subject': '机器学习（进阶）',
  'difficulty': 'hard',
  'method': '核心目标：掌握模型优化；学习步骤：1.特征工程 2.超参数调优；注意事项：研读论文复现实验'}]

In [29]:
from langchain_core.example_selectors import BaseExampleSelector
from langchain_core.example_selectors import LengthBasedExampleSelector
from typing import Dict, List


# 3. ExampleSelector：按长度筛选示例
# example_selector = LengthBasedExampleSelector(
#     examples=examples,
#     example_prompt=PromptTemplate(
#         input_variables=["subject", "difficulty", "method"],
#         template="学科：{subject}\n难度：{difficulty}\n学习方法：{method}\n"
#     ),
#     max_length=150,  # 控制示例总长度，避免提示词过长
#     get_text_length=lambda x: len(x)  # 长度计算函数
# )

# 3. 自定义ExampleSelector：按难度筛选示例（输入含difficulty参数）
class DifficultyExampleSelector(BaseExampleSelector):
    """根据用户输入的 difficulty 字段筛选样本"""
    def __init__(self, examples: List[Dict[str, str]]):
        self.examples = examples

    def add_example(self, example: Dict[str, str]) -> None:
        self.examples.append(example)

    def select_examples(self, input_variables: Dict[str, str]) -> List[Dict]:
        # 获取用户输入的难度等级，如果没有提供则默认为 'easy'
        target_difficulty = input_variables.get("difficulty", "easy")

        # 过滤出匹配难度的所有示例
        return [ex for ex in self.examples if ex.get("difficulty") == target_difficulty]


example_selector = DifficultyExampleSelector(examples=examples)

In [30]:
# 4. 构建工程化少样本模板
few_shot_prompt = FewShotPromptTemplate(
    example_selector=example_selector,  # 替换固定examples为动态选择器
    example_prompt=PromptTemplate(
        input_variables=["subject", "difficulty", "method"],
        template="学科：{subject}\n难度：{difficulty}\n学习方法：{method}\n"
    ),
    example_separator="\n",
    prefix="少样本提示：",
    suffix="参考以上示例，回答：\n学科：{new_subject}\n难度：{difficulty}\n学习方法：",
    input_variables=["new_subject", "difficulty"]  # 新增难度参数
)

In [31]:
# 5. 动态生成不同难度的提示词
# 场景1：生成入门级LangChain学习方法
formatted_prompt_easy = few_shot_prompt.format(
    new_subject="LangChain",
    difficulty="easy"
)

print(formatted_prompt_easy)


少样本提示：
学科：Python编程（入门）
难度：easy
学习方法：核心目标：掌握基础语法；学习步骤：1.变量与数据类型 2.条件语句；注意事项：边学边练

学科：机器学习（入门）
难度：easy
学习方法：核心目标：理解基础概念；学习步骤：1.数据预处理 2.简单模型；注意事项：用Excel辅助理解

参考以上示例，回答：
学科：LangChain
难度：easy
学习方法：


In [32]:
result_easy = chat_model.invoke([{"role": "user", "content": formatted_prompt_easy}])
print("\n入门级学习方法 (AI回复)：")
print(result_easy.content)


入门级学习方法 (AI回复)：
学科：LangChain  
难度：easy  
学习方法：核心目标：掌握链式调用与提示模板；学习步骤：1. 环境搭建与LLM调用 2. 提示模板与输出解析；注意事项：先跑通官方示例再修改参数


In [33]:
# 场景2：生成进阶级LangChain学习方法
formatted_prompt_hard = few_shot_prompt.format(
    new_subject="LangChain",
    difficulty="hard"
)

print(formatted_prompt_hard)


少样本提示：
学科：Python编程（进阶）
难度：hard
学习方法：核心目标：掌握面向对象与库开发；学习步骤：1.类与对象 2.模块开发；注意事项：参与开源项目

学科：机器学习（进阶）
难度：hard
学习方法：核心目标：掌握模型优化；学习步骤：1.特征工程 2.超参数调优；注意事项：研读论文复现实验

参考以上示例，回答：
学科：LangChain
难度：hard
学习方法：


In [34]:
result_hard = chat_model.invoke([{"role": "user", "content": formatted_prompt_hard}])
print("\n进阶级学习方法 (AI回复)：")
print(result_hard.content)


进阶级学习方法 (AI回复)：
学科：LangChain  
难度：hard  
学习方法：核心目标：掌握链式调用与智能体开发；学习步骤：1.提示模板与输出解析器 2.链与路由 3.记忆组件 4.工具调用与智能体构建；注意事项：深入源码理解回调机制，实践多模型集成与复杂工作流编排


## 2.3 输出解析：让输出更可控

#### 2.3.2.1 案例1：StrOutputParser

In [35]:
from langchain_core.output_parsers import StrOutputParser

# 1. 创建 StrOutputParser
# 核心作用：将 LLM 返回的 AIMessage 对象，统一转为纯字符串（str）
parser = StrOutputParser()

In [36]:
parser

StrOutputParser()

In [37]:
# 2. 链式调用：模型 → 字符串解析
chain = chat_model | parser
result = chain.invoke("请简要介绍 LangChain 输出解析层的作用")
print("StrOutputParser 解析后的字符串：")
print(result)
print("\n解析结果类型：", type(result))  # str

StrOutputParser 解析后的字符串：
LangChain 的输出解析层（Output Parsers）主要作用是**将大语言模型（LLM）返回的原始文本输出，转换为结构化的、程序可直接使用的数据格式**。

具体来说，它解决了以下三个核心问题：

1.  **结构化输出**：LLM 默认返回的是自然语言字符串。输出解析器可以将其解析为 Python 对象，如字典、列表、Pydantic 模型（数据类）或自定义对象，方便后续代码处理。

2.  **格式约束与重试**：当需要 LLM 输出特定格式（如 JSON、Markdown 表格、代码块）时，解析器可以：
    -   在 Prompt 中自动插入格式指令。
    -   如果 LLM 输出格式错误，解析器会抛出异常，并自动将错误信息反馈给 LLM，**要求其重新生成**，直到输出符合预期格式。

3.  **信息提取**：从 LLM 的冗长回复中，精准提取出关键字段（如日期、名称、数值），并自动进行类型转换（如字符串转数字）。

**常见类型举例：**

-   **`StrOutputParser`**：最简单的，直接返回字符串（通常用于聊天模型）。
-   **`CommaSeparatedListOutputParser`**：将“苹果, 香蕉, 橘子”解析为 `["苹果", "香蕉", "橘子"]`。
-   **`PydanticOutputParser`**：最常用。根据你定义的 Pydantic 数据模型（如 `class Person(BaseModel): name: str; age: int`），自动生成 Prompt 指令，并将 LLM 输出解析为 `Person` 对象。
-   **`JsonOutputParser`**：直接解析 JSON 格式的字符串。

**简单工作流程：**

1.  **定义解析器**（如 `PydanticOutputParser`）。
2.  **将解析器的格式指令注入 Prompt**（自动添加“请输出 JSON 格式...”）。
3.  **调用 LLM** 获得原始文本。
4.  **调用 `parser.parse()`** 得到结构化对象。

**总结：** 输出解析层是连接**自然语言**与**程序逻辑**的桥梁，确保 LL

#### 2.3.2.2 案例2：JsonOutputParser

In [39]:
from langchain_core.output_parsers import JsonOutputParser

# 1. 创建 JSON 解析器（无需额外配置，默认引导模型输出 JSON）
parser = JsonOutputParser()

In [40]:
from langchain_core.prompts import PromptTemplate

# 2. 构建提示模板（无需手动嵌入格式指令，解析器自动关联）
prompt = PromptTemplate(
    template="请介绍1个LangChain开发工具，输出工具名和核心功能。{format_instructions}",
    input_variables=[],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

In [41]:
# 3. 链式调用（LangChain ≥1.0.0 推荐方式，自动完成提示+调用+解析）
chain = prompt | chat_model | parser
result = chain.invoke({})  # 无输入参数，传入空字典

print("解析后的JSON（Python字典）：")
print(result)
print("获取单个字段：", result["tool_name"])  # 可直接用于业务逻辑

解析后的JSON（Python字典）：
{'tool_name': 'LangSmith', 'core_functionality': '用于调试、测试、评估和监控基于LangChain构建的LLM应用程序的全生命周期管理平台，支持追踪链的调用步骤、可视化运行日志、进行回归测试和性能分析。'}
获取单个字段： LangSmith


#### 2.3.2.3 案例3：PydanticOutputParser

In [43]:
from pydantic import BaseModel, Field

# 1. 定义 Pydantic 数据模型
class ToolInfo(BaseModel):
    tool_name: str = Field(description="开发工具的名称，如 LangSmith")
    function: str = Field(description="工具的核心功能，30字以内")
    difficulty: str = Field(description="学习难度，仅可选：简单 / 中等 / 复杂")

In [44]:
from langchain_core.output_parsers import PydanticOutputParser

# 2. 创建解析器
parser = PydanticOutputParser(pydantic_object=ToolInfo)

In [51]:
# 3. Prompt + Chain
prompt = PromptTemplate(
    template="{user_input}，严格按照要求输出。\n{format_instructions}",
    input_variables=["user_input"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

chain = prompt | chat_model | parser
result = chain.invoke({"user_input": "请介绍1个 Python 开发工具"})

In [46]:
print("解析后的结构化数据（Pydantic 模型对象）：")
print(result)

print("字段校验 difficulty：", result.difficulty)

print("转化为字典：", result.model_dump())

解析后的结构化数据（Pydantic 模型对象）：
tool_name='PyCharm' function='集成开发环境，支持代码编辑、调试、测试和版本控制' difficulty='中等'
字段校验 difficulty： 中等
转化为字典： {'tool_name': 'PyCharm', 'function': '集成开发环境，支持代码编辑、调试、测试和版本控制', 'difficulty': '中等'}


In [47]:
result

ToolInfo(tool_name='PyCharm', function='集成开发环境，支持代码编辑、调试、测试和版本控制', difficulty='中等')

In [48]:
chain = prompt | chat_model
result = chain.invoke({"user_input": "请介绍1个 Python 开发工具"})

In [50]:
result.content

'```json\n{\n  "tool_name": "PyCharm",\n  "function": "集成开发环境，支持代码编辑、调试、测试和版本控制",\n  "difficulty": "中等"\n}\n```'

### 2.3.3 BaseOutputParser 核心抽象接口

In [55]:
from langchain_core.output_parsers import BaseOutputParser

# 自定义解析器
class CustomToolParser(BaseOutputParser):
    def parse(self, text: str) -> dict:
        """将模型输出按 '工具名@核心功能@学习难度' 解析为字典"""
        text = text.strip().replace("\n", "").replace(" ", "")
        parts = text.split("@")
        if len(parts) != 3:
            raise ValueError(f"输出格式错误！需满足「工具名@核心功能@学习难度」，当前输出：{text}")
        return {
            "tool_name": parts[0].strip(),
            "function": parts[1].strip(),
            "difficulty": parts[2].strip()
        }

    def get_format_instructions(self) -> str:
        """生成提示词，引导模型按自定义格式输出"""
        return "请严格按照「工具名@核心功能@学习难度」格式输出，不添加多余内容。示例：LangSmith@全链路调试监控@中等"

In [56]:
# 使用解析器
parser = CustomToolParser()
prompt = PromptTemplate(
    template="请介绍1个LangChain开发工具。{format_instructions}",
    input_variables=[],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)
chain = prompt | chat_model | parser
result = chain.invoke({})

print("自定义解析器解析结果：")
print(result)
print("解析结果类型：", type(result))

自定义解析器解析结果：
{'tool_name': 'LangChainCLI', 'function': '快速创建项目模板', 'difficulty': '简单'}
解析结果类型： <class 'dict'>
